# local_policy

> Phase 12: `LocalPolicy`, a policy backed by an open-weight model (Hugging Face generation on cuda, the Apple GPU or cpu; vLLM on cuda), with an optional LoRA adapter and a reply cache; prompt encoding for training. GPU libraries are imported only when a model is used, so the offline tests need neither `torch` nor a model.

In [ ]:
#| default_exp local_policy

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib
import importlib.util
import itertools
import json
from pathlib import Path

from explRL.clue_policy import asReply, userText
from explRL.clue_text import CLUE_SCHEMA
from explRL.llm_client import ResponseCache

LORA_IDS = itertools.count(1)             # vLLM caches LoRA adapters by id: never reuse one

def pickDevice(pref='auto'):
    "cuda if available, else mps (the Apple GPU), else cpu; an explicit choice is kept."
    if pref != 'auto':
        return pref
    import torch
    if torch.cuda.is_available():
        return 'cuda'
    return 'mps' if torch.backends.mps.is_available() else 'cpu'


def dtypeFor(device):
    import torch
    return torch.float32 if device == 'cpu' else torch.bfloat16


def adapterHash(path):
    "sha256 over an adapter directory's weight files ('' without an adapter)."
    if not path:
        return ''
    h = hashlib.sha256()
    for p in sorted(Path(path).rglob('*')):
        if p.suffix in ('.safetensors', '.bin'):
            h.update(p.relative_to(path).as_posix().encode())
            h.update(p.read_bytes())
    return h.hexdigest()


def promptText(tok, system, blocks):
    "The chat-template text of a system message and one user message, with the generation prompt."
    return tok.apply_chat_template([dict(role='system', content=system), dict(role='user', content=userText(blocks))],
                                   tokenize=False, add_generation_prompt=True)


def encodePair(tok, prompt, reply, maxLen=1536, eos=True):
    "(ids, start): prompt and reply token ids (the reply ends with eos unless cut), the prompt cut from the left to fit."
    p = tok(prompt, add_special_tokens=False)['input_ids']
    r = tok(reply, add_special_tokens=False)['input_ids'] + ([tok.eos_token_id] if eos else [])
    p = p[max(0, len(p) + len(r) - maxLen):]
    return p + r, len(p)


def cacheKey(model, aHash, text, temperature, maxNew, seed, occurrence=0):
    return hashlib.sha256(json.dumps([model, aHash, text, float(temperature), int(maxNew), int(seed), int(occurrence)])
                          .encode()).hexdigest()


class LocalPolicy:
    """A policy backed by an open-weight model: Hugging Face batched generation (cuda, mps, cpu) or vLLM (cuda), an
    optional LoRA adapter, an optional reply cache (a ResponseCache directory). Libraries and weights load on first use;
    `hf` and `tokenizer` reuse a loaded (training) model, `adapterName` selects one of its adapters. guided (Phase 17, vLLM
    only): replies sampled under the reply JSON schema (structured outputs)."""
    def __init__(self, model, adapter=None, device='auto', maxNew=160, temperature=1.0, seed=0, cache=None,
                 engine='auto', gpuShare=0.35, batch=32, hf=None, tokenizer=None, adapterName=None, share=None, guided=False):
        self.model, self.maxNew, self.temperature, self.seed = model, int(maxNew), float(temperature), int(seed)
        self.device, self.gpuShare, self.batch, self.calls = pickDevice(device), float(gpuShare), int(batch), 0
        cuda = self.device == 'cuda' and importlib.util.find_spec('vllm') is not None
        self.engine = engine if engine != 'auto' else ('vllm' if cuda and hf is None else 'hf')
        self.cache = ResponseCache(cache) if cache else None
        self.hf, self.tokenizer, self.adapterName, self.llm, self.loraId = hf, tokenizer, adapterName, None, 0
        self.share = share                 # another LocalPolicy whose vLLM engine this one uses (per-role adapters)
        self.guided = bool(guided)
        if self.guided and self.engine != 'vllm':
            raise ValueError('guided (constrained) replies need the vLLM engine')
        self.setAdapter(adapter)

    def setAdapter(self, path):
        "Use the adapter at `path` from now on (vLLM: a new LoRA request; Hugging Face: loaded on first use)."
        self.adapter, self.aHash, self.loraId = path, adapterHash(path), next(LORA_IDS)   # unique across policies
        if self.hf is not None and path and self.engine == 'hf' and not hasattr(self.hf, 'peft_config'):
            from peft import PeftModel
            self.hf = PeftModel.from_pretrained(self.hf, path).eval()

    def tok(self):
        if self.tokenizer is None:
            from transformers import AutoTokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(self.model)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = 'left'
        return self.tokenizer

    def _hf(self):
        if self.hf is None:
            from transformers import AutoModelForCausalLM
            m = AutoModelForCausalLM.from_pretrained(self.model, dtype=dtypeFor(self.device)).to(self.device)
            if self.adapter:
                from peft import PeftModel
                m = PeftModel.from_pretrained(m, self.adapter)
            self.hf = m.eval()
        return self.hf

    def _stops(self, m):
        e = getattr(getattr(m, 'generation_config', None), 'eos_token_id', None)
        return set(e if isinstance(e, list) else [e]) | {self.tok().eos_token_id, self.tok().pad_token_id} - {None}

    def _genHF(self, texts):
        import torch
        tok, m, out = self.tok(), self._hf(), []
        if self.adapterName and hasattr(m, 'set_adapter'):
            m.set_adapter(self.adapterName)
        stops = self._stops(m)
        for a in range(0, len(texts), self.batch):
            enc = tok(texts[a:a + self.batch], return_tensors='pt', padding=True, add_special_tokens=False).to(self.device)
            torch.manual_seed(self.seed * 1_000_003 + self.calls * 10_007 + a)
            sample = dict(do_sample=True, temperature=self.temperature, top_p=1.0, top_k=0) if self.temperature > 0 else \
                dict(do_sample=False)
            with torch.no_grad():
                g = m.generate(**enc, max_new_tokens=self.maxNew, pad_token_id=tok.pad_token_id, **sample)
            n = enc['input_ids'].shape[1]
            for row, mask in zip(g[:, n:].tolist(), enc['attention_mask']):
                end = next((j for j, x in enumerate(row) if x in stops), None)
                ids = row if end is None else row[:end]
                out.append((tok.decode(ids, skip_special_tokens=True), int(mask.sum()), len(ids),
                            'max_tokens' if end is None and len(row) >= self.maxNew else 'end_turn'))
        return out

    def _engine(self):
        "The vLLM engine (created once; shared with `share`'s when given)."
        if self.llm is None:
            if self.share is not None:
                self.llm = self.share._engine()
            else:
                from vllm import LLM
                self.llm = LLM(self.model, enable_lora=True, max_lora_rank=64, gpu_memory_utilization=self.gpuShare,
                               seed=self.seed, dtype='bfloat16')
        return self.llm

    def _genVLLM(self, texts):
        from vllm import SamplingParams
        from vllm.lora.request import LoRARequest
        sps = [SamplingParams(temperature=self.temperature, top_p=1.0, max_tokens=self.maxNew,
                              seed=self.seed * 1_000_003 + self.calls * 10_007 + n, **self._structured()) for n in range(len(texts))]
        lr = LoRARequest(f'a{self.loraId}', self.loraId, self.adapter) if self.adapter else None
        res = self._engine().generate(texts, sps, lora_request=lr, use_tqdm=False)
        return [(o.outputs[0].text, len(o.prompt_token_ids), len(o.outputs[0].token_ids),
                 'max_tokens' if o.outputs[0].finish_reason == 'length' else 'end_turn') for o in res]

    def _structured(self):
        "SamplingParams keywords for guided replies (vLLM structured outputs; the older guided-decoding name as fallback)."
        if not self.guided:
            return {}
        try:
            from vllm.sampling_params import StructuredOutputsParams
            return dict(structured_outputs=StructuredOutputsParams(json=CLUE_SCHEMA))
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            return dict(guided_decoding=GuidedDecodingParams(json=CLUE_SCHEMA))

    def _generate(self, texts):
        "-> [(text, prompt tokens, reply tokens, stop reason)]; every request sampled independently."
        return self._genVLLM(texts) if self.engine == 'vllm' else self._genHF(texts)

    def __call__(self, requests):
        texts = [promptText(self.tok(), s, b) for s, b in requests]
        seen, keys = {}, []
        for t in texts:                    # repeated prompts in one call (a GRPO group) are distinct samples
            seen[t] = seen.get(t, -1) + 1
            keys.append(cacheKey(self.model + '|guided' * self.guided, self.aHash, t, self.temperature, self.maxNew,
                                 self.seed + self.calls, seen[t]))
        got = [self.cache.get(k) if self.cache else None for k in keys]
        miss = [n for n, g in enumerate(got) if g is None]
        for n, r in zip(miss, self._generate([texts[n] for n in miss]) if miss else []):
            v = dict(text=r[0], nIn=r[1], nOut=r[2], stop=r[3])
            got[n] = self.cache.put(keys[n], v) if self.cache else v
        self.calls += 1
        return [asReply(g['text'], dict(input_tokens=g['nIn'], output_tokens=g['nOut']), g['stop']) for g in got]

## Tests

In [ ]:
import json, tempfile
from pathlib import Path
from explRL.local_policy import *
from explRL.clue_policy import userText

class StubTok:
    "A stand-in tokenizer: one id per character, eos 0; the chat template is the JSON of the messages."
    eos_token_id, pad_token, eos_token = 0, '<pad>', '<eos>'
    def __call__(self, text, add_special_tokens=False):
        return dict(input_ids=[ord(c) for c in text])
    def apply_chat_template(self, msgs, tokenize=False, add_generation_prompt=True):
        return json.dumps(msgs)

class Echo(LocalPolicy):
    "Generation replaced by a counter, to test caching and the request bookkeeping without a model."
    n = 0
    def _generate(self, texts):
        out = []
        for t in texts:
            Echo.n += 1
            out.append((json.dumps(dict(action='wait', notes=f'g{Echo.n}')), len(t), 5, 'end_turn'))
        return out

assert pickDevice('cpu') == 'cpu' and pickDevice('mps') == 'mps'
blocks = [{'type': 'text', 'text': 'Your notes so far: a'}, {'type': 'text', 'text': 'Now:\nobs'}]
assert json.loads(promptText(StubTok(), 'sys', blocks)) == [dict(role='system', content='sys'), dict(role='user', content=userText(blocks))]
# the reply (plus eos) is kept whole; the prompt is cut from the left to fit
ids, start = encodePair(StubTok(), 'abcdef', 'xy', maxLen=6)
assert ids == [ord(c) for c in 'defxy'] + [0] and start == 3
ids, start = encodePair(StubTok(), 'ab', 'xy', maxLen=50)
assert ids == [ord(c) for c in 'abxy'] + [0] and start == 2
assert encodePair(StubTok(), 'ab', 'xy', eos=False)[0] == [ord(c) for c in 'abxy']
assert len({cacheKey('m', '', 't', 1.0, 160, 0, 0), cacheKey('m', '', 't', 1.0, 160, 0, 1), cacheKey('m', 'h', 't', 1.0, 160, 0, 0)}) == 3
with tempfile.TemporaryDirectory() as d:
    assert adapterHash(None) == '' and adapterHash(d) == adapterHash(d)
    (Path(d) / 'adapter_model.safetensors').write_bytes(b'1')
    h1 = adapterHash(d)
    (Path(d) / 'adapter_model.safetensors').write_bytes(b'2')
    assert h1 != adapterHash(d)
reqs = [('sys', blocks), ('sys', blocks), ('sys', [{'type': 'text', 'text': 'Now:\nother'}])]
with tempfile.TemporaryDirectory() as c:
    a = Echo('m', device='cpu', engine='hf', tokenizer=StubTok(), cache=c)(reqs)
    assert len({x.parsed['notes'] for x in a}) == 3              # a repeated prompt is a separate sample
    assert a[0].usage['output_tokens'] == 5 and a[0].cost == 0 and a[0].stopReason == 'end_turn'
    b = Echo('m', device='cpu', engine='hf', tokenizer=StubTok(), cache=c)(reqs)
    assert [x.text for x in a] == [x.text for x in b] and Echo.n == 3     # the second run is served from the cache
    Echo('m', device='cpu', engine='hf', tokenizer=StubTok(), cache=c, seed=1)(reqs)
    assert Echo.n == 6                                            # another sampling seed is another key
assert Echo('m', device='cpu', engine='auto', tokenizer=StubTok()).engine == 'hf'
x, y = Echo('m', device='cpu', tokenizer=StubTok()), Echo('m', device='cpu', tokenizer=StubTok())
x.setAdapter(None); y.setAdapter(None)
assert len({x.loraId, y.loraId}) == 2
# Phase 17: guided replies need vLLM (the HF engine refuses them); off by default, so keys and parameters are unchanged
try:
    LocalPolicy('m', device='cpu', guided=True)
    raise AssertionError('the HF engine must refuse guided replies')
except ValueError:
    pass
lp = LocalPolicy('m', device='cpu')
assert lp.guided is False and lp._structured() == {}

In [ ]:
#| train
# With the [train] extra, no download: generation on this machine's device with the tiny random model
import os
if os.environ.get('IN_TEST'):   # under nbdev-test only: the docs build runs cells with imports and has no [train] extra
    import tempfile
    from explRL.clue_agents import ClueBoard
    from explRL.clue_policy import renderPrompt
    from explRL.clue_text import TextClueEnv
    from explRL.clueboard import ClueBoardEnv
    from explRL.tiny_model import tinyModel
    with tempfile.TemporaryDirectory() as d:
        tiny = tinyModel(d)
        env = TextClueEnv(ClueBoardEnv(1, 4, m=2, r=1, R=30))
        for dev in ('cpu', pickDevice()):
            out = LocalPolicy(tiny, maxNew=16, device=dev)([renderPrompt(env, env.observe(0, 0.0, ClueBoard()))] * 4)
            assert len({o.text for o in out}) > 1 and all(o.usage['input_tokens'] > 100 for o in out)   # samples differ
        with tempfile.TemporaryDirectory() as c:
            reqs = [renderPrompt(env, env.observe(i, 0.0, ClueBoard())) for i in range(4)]
            a = [o.text for o in LocalPolicy(tiny, maxNew=12, device='cpu', cache=c)(reqs)]
            assert a == [o.text for o in LocalPolicy(tiny, maxNew=12, device='cpu', cache=c)(reqs)]          # cached replay

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()